# 01: micrograd — 自動微分エンジンをスクラッチ実装

**参照**: Karpathy "Neural Networks: Zero to Hero" 第1回  
**動画**: https://youtu.be/VMj-3S1tku0  
**GitHub**: https://github.com/karpathy/micrograd

## 目標
- 自動微分（Autograd）の仕組みを理解する
- `Value` クラスを自分で実装して PyTorch と同じ挙動を再現する
- 連鎖律（Chain Rule）がコードでどう表現されるか体得する

## Step 0: 数値微分で確認（直感を作る）

微分の定義: `df/dx = lim(h→0) [f(x+h) - f(x)] / h`

In [1]:
def f(x):
    return 3*x**2 - 4*x + 5

# 数値微分
x = 3.0
h = 0.0001
numerical_grad = (f(x + h) - f(x)) / h
analytical_grad = 6*x - 4  # 解析的な微分

print(f"f(x)  at x=3: {f(x)}")
print(f"数値微分: {numerical_grad:.6f}")
print(f"解析微分: {analytical_grad:.6f}")

f(x)  at x=3: 20.0
数値微分: 14.000300
解析微分: 14.000000


## Step 1: Value クラスの骨格

各数値が「自分がどう計算されたか」を覚えている設計

In [2]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0          # 勾配（初期値0）
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op           # どの演算で生まれたか
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    # --- 演算の実装（ここを埋めていく） ---

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            # c = a + b → dc/da = 1, dc/db = 1
            # 連鎖律: self.grad += 1.0 * out.grad
            self.grad  += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            # c = a * b → dc/da = b, dc/db = a
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float))
        out = Value(self.data ** exponent, (self,), f'**{exponent}')

        def _backward():
            # c = a^n → dc/da = n * a^(n-1)
            self.grad += (exponent * self.data**(exponent - 1)) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0, self.data), (self,), 'ReLU')

        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        import math
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')

        def _backward():
            # d(tanh(x))/dx = 1 - tanh(x)^2
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    # 逆方向演算（右辺に来たとき）
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __neg__(self):         return self * -1
    def __sub__(self, other):  return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __truediv__(self, other): return self * other**-1

    def backward(self):
        """トポロジカルソートして逆順に勾配を伝播"""
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.0   # dL/dL = 1
        for node in reversed(topo):
            node._backward()

print("Value クラス定義完了")

Value クラス定義完了


## Step 2: 動作確認 — 手計算と照合

In [3]:
# 式: L = (a * b + c) * f
# 手計算: a=2, b=-3, c=10, f=-2
#   e = a*b = -6
#   d = e+c = 4
#   L = d*f = -8
#
#   dL/dd = f = -2
#   dL/dc = dL/dd * dd/dc = -2 * 1 = -2
#   dL/de = -2
#   dL/da = dL/de * de/da = -2 * b = -2 * -3 = 6
#   dL/db = dL/de * de/db = -2 * a = -2 *  2 = -4
#   dL/df = d = 4

a = Value(2.0,  label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
f = Value(-2.0, label='f')

e = a * b;       e.label = 'e'
d = e + c;       d.label = 'd'
L = d * f;       L.label = 'L'

L.backward()

print(f"L = {L.data}  (期待: -8.0)")
print(f"a.grad = {a.grad}  (期待: 6.0)")
print(f"b.grad = {b.grad}  (期待: -4.0)")
print(f"c.grad = {c.grad}  (期待: -2.0)")
print(f"f.grad = {f.grad}  (期待: 4.0)")

L = -8.0  (期待: -8.0)
a.grad = 6.0  (期待: 6.0)
b.grad = -4.0  (期待: -4.0)
c.grad = -2.0  (期待: -2.0)
f.grad = 4.0  (期待: 4.0)


## Step 3: PyTorch と比較

In [4]:
import torch

a_t = torch.tensor(2.0,  requires_grad=True)
b_t = torch.tensor(-3.0, requires_grad=True)
c_t = torch.tensor(10.0, requires_grad=True)
f_t = torch.tensor(-2.0, requires_grad=True)

L_t = (a_t * b_t + c_t) * f_t
L_t.backward()

print("PyTorch の結果:")
print(f"a.grad = {a_t.grad.item()}")
print(f"b.grad = {b_t.grad.item()}")
print(f"c.grad = {c_t.grad.item()}")
print(f"f.grad = {f_t.grad.item()}")
print("\n↑ micrograd の結果と一致していれば成功！")

PyTorch の結果:
a.grad = 6.0
b.grad = -4.0
c.grad = -2.0
f.grad = 4.0

↑ micrograd の結果と一致していれば成功！


## Step 4: 1ニューロンの実装

In [5]:
import random

class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


# 動作確認
x = [2.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1])  # 3入力 → 4 → 4 → 1出力
out = n(x)
print(f"出力: {out}")
print(f"パラメータ数: {len(n.parameters())}")

出力: Value(data=0.8481, grad=0.0000)
パラメータ数: 41


## Step 5: 簡単な学習ループ

In [6]:
# XOR っぽいデータセット
xs = [
    [2.0,  3.0, -1.0],
    [3.0, -1.0,  0.5],
    [0.5,  1.0,  1.0],
    [1.0,  1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]  # 期待する出力

model = MLP(3, [4, 4, 1])

for k in range(20):
    # forward
    ypred = [model(x) for x in xs]
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))

    # backward
    for p in model.parameters():
        p.grad = 0.0   # 勾配リセット（重要！）
    loss.backward()

    # update
    for p in model.parameters():
        p.data -= 0.1 * p.grad

    if k % 5 == 0 or k == 19:
        print(f"step {k:02d} | loss = {loss.data:.6f}")

print("\n予測値:")
for x, y in zip(xs, ys):
    pred = model(x).data
    print(f"  入力{x} → 予測 {pred:.4f}  (正解 {y})")

step 00 | loss = 4.547424
step 05 | loss = 0.124626
step 10 | loss = 0.045372
step 15 | loss = 0.029239
step 19 | loss = 0.022801

予測値:
  入力[2.0, 3.0, -1.0] → 予測 0.9261  (正解 1.0)
  入力[3.0, -1.0, 0.5] → 予測 -0.9321  (正解 -1.0)
  入力[0.5, 1.0, 1.0] → 予測 -0.9127  (正解 -1.0)
  入力[1.0, 1.0, -1.0] → 予測 0.9374  (正解 1.0)


## 確認問題（動画セッションごとに答えを書いてみよう）

1. `+=` が必要で `=` ではダメな理由を `a*a` を例に説明せよ
2. トポロジカルソートが必要な理由を一文で
3. `__radd__` が必要になる場面は？
4. `tanh` の微分が `1 - tanh(x)^2` になる理由を紙で導出せよ

---
次のステップ → `../02_makemore_bigram/`

In [1]:
from micrograd.engine import Value

# === Step 1: パラメータ（学習される数字）===
w = Value(2.0)   # 重み（最初は適当な値）
b = Value(-1.0)  # バイアス

# === Step 2: 入力と正解 ===
x = Value(3.0)   # 入力データ
y_true = 10.0    # 正解

# === Step 3: 予測（forward） ===
y_pred = w * x + b   # 2.0 * 3.0 + (-1.0) = 5.0

# === Step 4: 損失（誤差の大きさ）===
loss = (y_pred - y_true) ** 2   # (5 - 10)^2 = 25

print(f"予測値: {y_pred.data}")
print(f"損失:   {loss.data}")

# === Step 5: 勾配を計算（backward）===
loss.backward()

print(f"\n損失を減らすには...")
print(f"w.grad = {w.grad}  ← wをこの逆方向に動かせばいい")
print(f"b.grad = {b.grad}  ← bをこの逆方向に動かせばいい")

# === Step 6: パラメータを更新 ===
lr = 0.01  # 学習率（どれだけ動かすか）
w.data -= lr * w.grad
b.data -= lr * b.grad

print(f"\n更新後: w={w.data:.3f}, b={b.data:.3f}")

# === 更新後の予測 ===
y_pred2 = w.data * 3.0 + b.data
print(f"更新後の予測値: {y_pred2:.3f}  (正解: {y_true})")


ModuleNotFoundError: No module named 'micrograd'